In [50]:
!pip install fastapi uvicorn pydantic email-validator pandas plotly -q
print("✅ Librerías instaladas y listas.")

✅ Librerías instaladas y listas.


In [51]:
import threading
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, EmailStr
from typing import Optional

app = FastAPI()

class OportunidadVenta(BaseModel):
    id_oportunidad: int
    estado: str
    portafolio: str
    servicio: str
    frente: str
    nombre_cliente: str
    documento_cliente: int
    email_contacto: EmailStr
    cel_contact: str
    contrato: Optional[float] = None
    permanencia: Optional[str] = None

@app.post("/api/oportunidades/")
def crear_oportunidad(oportunidad: OportunidadVenta):
    alerta_financiera = None

    # El Agente evalúa si el cliente canceló para lanzar la estrategia de rescate
    if oportunidad.estado == "Anulado":
        if oportunidad.servicio == "AVANZADO":
            descuento_sugerido = "25%"
            plan_alternativo = "BÁSICO"
            prioridad = "ALTA 🚨"
        else:
            descuento_sugerido = "15%"
            plan_alternativo = "Plan Retención Pyme"
            prioridad = "MEDIA ⚠️"

        valor_contrato = f"${oportunidad.contrato:,.0f}" if oportunidad.contrato else "No especificado"
        alerta_financiera = f"[{prioridad}] Contrato: {valor_contrato}. Sugerencia: Migrar a {plan_alternativo} con {descuento_sugerido} de desc."

    return {"status": "ok", "analisis_ia": alerta_financiera}

# Lanzar el servidor en segundo plano de manera limpia
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 ¡Servidor FastAPI corriendo con éxito en segundo plano en el Puerto 8000!")

🚀 ¡Servidor FastAPI corriendo con éxito en segundo plano en el Puerto 8000!


ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


In [52]:
import pandas as pd
import requests
import time
import plotly.express as px
from IPython.display import display, HTML

API_URL = "http://127.0.0.1:8000/api/oportunidades/"

def simular_crm_y_graficar(ruta_csv):
    print("🚀 Iniciando lectura de datos...")
    df = pd.read_csv(ruta_csv)
    df.columns = [col.upper() for col in df.columns]

    contador_exitos = 0
    contador_anulados = 0
    alertas_ia = []

    # Procesar fila por fila enviando a la API
    for index, row in df.iterrows():
        email_original = str(row.get('EMAIL_CONTACTO', '')).strip()
        email_limpio = email_original if pd.notnull(row.get('EMAIL_CONTACTO')) and "@" in email_original else "sin_correo@crm.com"

        contrato_val = None
        if pd.notnull(row.get('CONTRATO')):
            try: contrato_val = float(row['CONTRATO'])
            except ValueError: pass

        payload = {
            "id_oportunidad": int(row['ID']) if pd.notnull(row.get('ID')) else int(index),
            "estado": str(row.get('ESTADO', 'Desconocido')),
            "portafolio": str(row.get('PORTAFOLIO', 'FIJO')),
            "servicio": str(row.get('SERVICIO', 'BASICO')),
            "frente": str(row.get('FRENTE', 'B2B')),
            "nombre_cliente": str(row.get('NOMBRE_CLIENTE', 'Cliente Anonimo')),
            "documento_cliente": int(row['DOCUMENTO_CLIENTE']) if pd.notnull(row.get('DOCUMENTO_CLIENTE')) else 0,
            "email_contacto": email_limpio,
            "cel_contact": str(row.get('CEL_CONTACTO', '0')),
            "contrato": contrato_val,
            "permanencia": str(row.get('PERMANENCIA')) if pd.notnull(row.get('PERMANENCIA')) else None
        }

        try:
            response = requests.post(API_URL, json=payload)
            if response.status_code == 200:
                contador_exitos += 1
                res = response.json()
                if res.get("analisis_ia"):
                    contador_anulados += 1
                    print(f"⚠️ Fila {index} ({payload['nombre_cliente']}): {res['analisis_ia']}")
        except requests.exceptions.ConnectionError:
            print("🚨 Error de conexión con la API.")
            return

    # Renderizar Métricas Visuales
    porcentaje = (contador_anulados / contador_exitos) * 100 if contador_exitos > 0 else 0
    html_cards = f"""
    <div style="font-family: Arial; display: flex; justify-content: space-around; background-color: #f8f9fa; padding: 15px; border-radius: 10px; margin: 20px 0; box-shadow: 0px 2px 4px rgba(0,0,0,0.1);">
        <div style="text-align: center;"><h4 style="margin:0;color:#6c757d;">Leads Cargados</h4><h2 style="margin:5px 0;">{contador_exitos} 👤</h2></div>
        <div style="text-align: center;"><h4 style="margin:0;color:#dc3545;">Casos en Riesgo</h4><h2 style="margin:5px 0;color:#dc3545;">{contador_anulados} ⚠️ ({porcentaje:.1f}%)</h2></div>
    </div>
    """
    display(HTML(html_cards))

    # Gráfico 1: Estado de los Leads
    fig_estado = px.histogram(df, x="ESTADO", color="ESTADO", title="📊 Estado Actual de la Base de Datos", template="plotly_white", width=800, height=350)
    fig_estado.show()

    # Guardar reporte filtrado en la carpeta lateral
    df_anulados = df[df['ESTADO'] == 'Anulado'].copy()
    df_anulados.to_csv("reporte_retencion_ia.csv", index=False)
    print("📂 Archivo 'reporte_retencion_ia.csv' guardado en tu panel izquierdo.")

# Ejecutar pasándole tu archivo
simular_crm_y_graficar("Somostelser.csv")

🚀 Iniciando lectura de datos...
⚠️ Fila 1 (CECILIA  ALICIA BRAVO DE NOHAVA): Alerta de riesgo: El cliente CECILIA  ALICIA BRAVO DE NOHAVA canceló un servicio BASICO. Agente IA recomienda contactar de inmediato para retención.
⚠️ Fila 8 (CEMAT SALUD SAS): Alerta de riesgo: El cliente CEMAT SALUD SAS canceló un servicio BASICO. Agente IA recomienda contactar de inmediato para retención.
⚠️ Fila 14 (INSTITUCIÓN DE SEGURIDAD Y SALUD ISESALUD SAS): Alerta de riesgo: El cliente INSTITUCIÓN DE SEGURIDAD Y SALUD ISESALUD SAS canceló un servicio BASICO. Agente IA recomienda contactar de inmediato para retención.
⚠️ Fila 21 (TRADING COLLEGE S.A.S.): Alerta de riesgo: El cliente TRADING COLLEGE S.A.S. canceló un servicio BASICO. Agente IA recomienda contactar de inmediato para retención.
⚠️ Fila 24 (EDILCE PADIERNA GIRALDO): Alerta de riesgo: El cliente EDILCE PADIERNA GIRALDO canceló un servicio BASICO. Agente IA recomienda contactar de inmediato para retención.
⚠️ Fila 30 (CONECTIVIDAD CONTEL S

📂 Archivo 'reporte_retencion_ia.csv' guardado en tu panel izquierdo.


In [55]:
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, HTML

class CRMInteligente:
    def __init__(self, ruta_csv):
        self.ruta_csv = ruta_csv
        self.df = None

    def procesar_pipeline(self):
        # 1. Extracción y Limpieza de Datos (Python & Pandas)
        self.df = pd.read_csv(self.ruta_csv)
        self.df.columns = [col.upper() for col in self.df.columns]

        if 'CONTRATO' in self.df.columns:
            self.df['CONTRATO'] = pd.to_numeric(self.df['CONTRATO'], errors='coerce').fillna(0)

        # 2. Simulación del AGENTE FINANCIERO COGNITIVO
        # Este método actúa como el cerebro que n8n consultaría en producción
        def auditar_cuenta(fila):
            estado = str(fila['ESTADO']).strip().upper()
            servicio = str(fila['SERVICIO']).upper()
            contrato = fila['CONTRATO']

            if estado == 'ANULADO':
                if 'AVANZADO' in servicio:
                    return {
                        'NIVEL_RIESGO': '🚨 CRÍTICO (ALTA FUGA)',
                        'ACCION_FINANCIERA': f'Fuga de cuenta corporativa premium. Mitigar ofreciendo downgrade a plan BÁSICO con un 25% de descuento estructural sobre la tarifa actual de ${contrato:,.0f}.'
                    }
                else:
                    return {
                        'NIVEL_RIESGO': '⚠️ MODERADO',
                        'ACCION_FINANCIERA': f'Cuenta PYME estándar. Aplicar Plan Retención Pyme con 15% de descuento o congelamiento de facturación por 90 días.'
                    }
            return {'NIVEL_RIESGO': '✅ ESTABLE', 'ACCION_FINANCIERA': 'Cliente activo. Monitoreo predictivo regular.'}

        # Inyección de la analítica del Agente en el DataFrame del CRM
        resultados_agente = self.df.apply(auditar_cuenta, axis=1)
        self.df['RIESGO_IA'] = [r['NIVEL_RIESGO'] for r in resultados_agente]
        self.df['RECOMENDACION_AGENTE'] = [r['ACCION_FINANCIERA'] for r in resultados_agente]

        # Guardar base de datos maestra para la integración con n8n / Streamlit
        self.df.to_csv("crm_sistema_maestro.csv", index=False)

    def desplegar_dashboard_ejecutivo(self):
        total_clientes = len(self.df)
        clientes_anulados = len(self.df[self.df['ESTADO'] == 'Anulado'])
        churn_rate = (clientes_anulados / total_clientes) * 100

        # Renderizado de Tarjetas Métricas de IA en HTML
        html_cards = f"""
        <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; display: flex; justify-content: space-around; background-color: #0f172a; padding: 20px; border-radius: 12px; margin: 15px 0; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="text-align: center; color: #94a3b8;"><h4 style="margin:0; text-transform: uppercase; font-size:12px;">Muestra Total CRM</h4><h2 style="margin:5px 0; color: #f8fafc;">{total_clientes} Leads</h2></div>
            <div style="text-align: center; color: #f87171;"><h4 style="margin:0; text-transform: uppercase; font-size:12px;">Auditorías Agente Financiero</h4><h2 style="margin:5px 0; color: #ef4444;">{clientes_anulados} Alertas de Rescate</h2></div>
            <div style="text-align: center; color: #38bdf8;"><h4 style="margin:0; text-transform: uppercase; font-size:12px;">Tasa de Churn Global</h4><h2 style="margin:5px 0; color: #0ea5e9;">{churn_rate:.1f}%</h2></div>
        </div>
        """
        display(HTML(html_cards))

        # Gráfico analítico del Estado del CRM para la sustentación
        fig = px.histogram(self.df, x="ESTADO", color="RIESGO_IA",
                           title="Análisis de Riesgo del Ecosistema CRM - Somostelser",
                           template="plotly_dark",
                           color_discrete_map={'🚨 CRÍTICO (ALTA FUGA)': '#ef4444', '⚠️ MODERADO': '#f59e0b', '✅ ESTABLE': '#10b981'})

        # CORRECCIÓN AQUÍ: Se cambia 'bar_mode' por 'barmode'
        fig.update_layout(barmode='stack', title_font_size=18)
        fig.show()

# --- INSTANCIACIÓN DEL SISTEMA ---
crm_sistema = CRMInteligente("Somostelser.csv")
crm_sistema.procesar_pipeline()
crm_sistema.desplegar_dashboard_ejecutivo()

In [57]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# 1. Carga y preparación de la base maestra de Somostelser
df_maestro = pd.read_csv("crm_sistema_maestro.csv")

# Instanciar el contenedor de salida interactiva
output_crm = widgets.Output()

# 2. Componentes de la Interfaz (Filtros del CRM corporativo)
txt_buscar = widgets.Text(placeholder="Escribe el cliente B2B...", description="🔍 Cliente:")
drop_estado = widgets.Dropdown(options=['TODOS'] + list(df_maestro['ESTADO'].unique()), value='TODOS', description="📍 Estado:")
drop_servicio = widgets.Dropdown(options=['TODOS'] + list(df_maestro['SERVICIO'].unique()), value='TODOS', description="💼 Servicio:")

def refrescar_vista_crm(*args):
    with output_crm:
        clear_output(wait=True)

        # Filtrado dinámico en tiempo real
        df_temp = df_maestro.copy()
        if txt_buscar.value:
            df_temp = df_temp[df_temp['NOMBRE_CLIENTE'].str.contains(txt_buscar.value, case=False, na=False)]
        if drop_estado.value != 'TODOS':
            df_temp = df_temp[df_temp['ESTADO'] == drop_estado.value]
        if drop_servicio.value != 'TODOS':
            df_temp = df_temp[df_temp['SERVICIO'] == drop_servicio.value]

        # Banner de Presentación Corporativa
        total_vista = len(df_temp)
        alertas_vista = len(df_temp[df_temp['ESTADO'] == 'Anulado'])

        display(HTML(f"""
        <div style="background-color: #0f172a; color: #f8fafc; padding: 15px; border-radius: 10px; font-family: sans-serif; margin-bottom: 15px; border-left: 5px solid #0ea5e9;">
            <h2 style="margin: 0; color: #38bdf8; font-size: 20px;">🏢 Sistema CRM Inteligente - Somostelser S.A.S.</h2>
            <p style="margin: 5px 0 0 0; font-size: 14px; color: #94a3b8;">
                Registros en pantalla: <b>{total_vista}</b> | Acciones requeridas por el Agente Financiero: <span style="color: #ef4444; font-weight: bold;">{alertas_vista}</span>
            </p>
        </div>
        """))

        # Mostrar la tabla del CRM priorizando las alertas de la IA
        columnas_show = ['ID', 'NOMBRE_CLIENTE', 'ESTADO', 'SERVICIO', 'CONTRATO', 'RIESGO_IA', 'RECOMENDACION_AGENTE']
        display(df_temp[columnas_show].head(10))

# Monitorear interacciones del usuario
txt_buscar.observe(refrescar_vista_crm, 'value')
drop_estado.observe(refrescar_vista_crm, 'value')
drop_servicio.observe(refrescar_vista_crm, 'value')

# Organizar visualmente el layout del CRM en pantalla
caja_filtros = widgets.HBox([txt_buscar, drop_estado, drop_servicio])
app_crm = widgets.VBox([caja_filtros, output_crm])

# Mostrar la aplicación interactiva
display(app_crm)
refrescar_vista_crm()

In [59]:
%%writefile crm_somostelser.py
import streamlit as st
import pandas as pd
import plotly.express as px

# 1. Configuración de Marca y Estilo de la Aplicación Corporativa
st.set_page_config(
    page_title="CRM Inteligente - Somostelser S.A.S.",
    page_icon="🏢",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Estilo personalizado para un look profesional moderno
st.markdown("""
    <style>
    .main-title { font-size: 32px; font-weight: bold; color: #0ea5e9; margin-bottom: 5px; }
    .subtitle { font-size: 16px; color: #64748b; margin-bottom: 25px; }
    .metric-card { background-color: #f8fafc; border-left: 5px solid #0ea5e9; padding: 15px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }
    </style>
""", unsafe_allow_html=True)

# 2. Carga Segura de Datos Optimizado para Streamlit
@st.cache_data
def cargar_datos_maestros():
    try:
        # Intentamos leer la base de datos ya procesada por el Agente Financiero
        df = pd.read_csv("crm_sistema_maestro.csv")
    except FileNotFoundError:
        # Respaldo en caso de leer el archivo base original
        df = pd.read_csv("Somostelser.csv")
        df.columns = [col.upper() for col in df.columns]
        if 'CONTRATO' in df.columns:
            df['CONTRATO'] = pd.to_numeric(df['CONTRATO'], errors='coerce').fillna(0)

        # Lógica del Agente Financiero integrada en el backend del frontend
        def auditar(fila):
            if str(fila.get('ESTADO')).strip().upper() == 'ANULADO':
                servicio = str(fila.get('SERVICIO')).upper()
                contrato = fila.get('CONTRATO')
                if 'AVANZADO' in servicio:
                    return ('🚨 CRÍTICO', f'Downgrade a plan BÁSICO con 25% de desc. sobre tarifa de ${contrato:,.0f}')
                return ('⚠️ MODERADO', 'Plan Retención Pyme con 15% de desc. o congelación de tarifa.')
            return ('✅ ESTABLE', 'Cliente activo. Monitoreo regular.')

        auditoria = df.apply(auditar, axis=1)
        df['RIESGO_IA'] = [r[0] for r in auditoria]
        df['RECOMENDACION_AGENTE'] = [r[1] for r in auditoria]
    return df

df_crm = cargar_datos_maestros()

# 3. Panel Lateral: Buscador y Filtros Interactivos del CRM
st.sidebar.image("https://via.placeholder.com/150x50.png?text=Somostelser", use_container_width=True)
st.sidebar.markdown("### 🔍 Panel de Búsqueda B2B")

buscar_cliente = st.sidebar.text_input("Buscar por Nombre de Cliente:")
filtro_estado = st.sidebar.selectbox("Filtrar por Estado:", ['TODOS'] + list(df_crm['ESTADO'].unique()))
filtro_servicio = st.sidebar.selectbox("Filtrar por Servicio:", ['TODOS'] + list(df_crm['SERVICIO'].unique()))

# Aplicar las consultas del usuario a la base del CRM en tiempo real
df_filtrado = df_crm.copy()
if buscar_cliente:
    df_filtrado = df_filtrado[df_filtrado['NOMBRE_CLIENTE'].str.contains(buscar_cliente, case=False, na=False)]
if filtro_estado != 'TODOS':
    df_filtrado = df_filtrado[df_filtrado['ESTADO'] == filtro_estado]
if filtro_servicio != 'TODOS':
    df_filtrado = df_filtrado[df_filtrado['SERVICIO'] == filtro_servicio]

# 4. Panel Principal: Interfaz Gráfica de Usuario (GUI)
st.markdown('<p class="main-title">🏢 Sistema CRM Inteligente — Somostelser S.A.S.</p>', unsafe_allow_html=True)
st.markdown('<p class="subtitle">Módulo de Auditoría Comercial Integrado con Agente Financiero Autónomo</p>', unsafe_allow_html=True)

# Fila de Indicadores Clave de Rendimiento (KPIs)
total_leads = len(df_filtrado)
alertas_criticas = len(df_filtrado[df_filtrado['ESTADO'] == 'Anulado'])

col_kpi1, col_kpi2, col_kpi3 = st.columns(3)
with col_kpi1:
    st.markdown(f'<div class="metric-card"><h4>Muestra en Pantalla</h4><h2>{total_leads} Leads</h2></div>', unsafe_allow_html=True)
with col_kpi2:
    st.markdown(f'<div class="metric-card" style="border-left-color: #ef4444;"><h4>Alertas del Agente</h4><h2>{alertas_criticas} Casos Riesgo</h2></div>', unsafe_allow_html=True)
with col_kpi3:
    st.markdown(f'<div class="metric-card" style="border-left-color: #10b981;"><h4>Estado Core IA</h4><h2>Activo (Local)</h2></div>', unsafe_allow_html=True)

st.markdown("---")

# Visualización del Pipeline en el CRM
st.subheader("📊 Distribución Analítica de las Cuentas")
fig_pipeline = px.histogram(
    df_filtrado, x="ESTADO", color="RIESGO_IA",
    template="plotly_white", barmode="stack",
    color_discrete_map={'🚨 CRÍTICO': '#ef4444', '⚠️ MODERADO': '#f59e0b', '✅ ESTABLE': '#10b981'}
)
fig_pipeline.update_layout(height=350, margin=dict(l=0, r=0, t=30, b=0))
st.plotly_chart(fig_pipeline, use_container_width=True)

# Tabla de Datos Corporativos Dinámica
st.subheader("📋 Consola de Gestión de Clientes B2B")
columnas_visibles = ['ID', 'NOMBRE_CLIENTE', 'ESTADO', 'SERVICIO', 'CONTRATO', 'RIESGO_IA', 'RECOMENDACION_AGENTE']
st.dataframe(df_filtrado[columnas_visibles], use_container_width=True, hide_index=True)

# Botón de Descarga Inmediata para el Equipo de Retención
csv_data = df_filtrado[columnas_visibles].to_csv(index=False).encode('utf-8')
st.sidebar.markdown("---")
st.sidebar.download_button(
    label="📥 Descargar Reporte Comercial",
    data=csv_data,
    file_name="crm_reporte_somostelser.csv",
    mime="text/csv"
)

Overwriting crm_somostelser.py


In [60]:
%%writefile requirements.txt
streamlit==1.35.0
pandas
plotly

Writing requirements.txt
